# 17 — Fannie external model reliability

Evaluate the four frozen companion models from the reviewed Notebook 16 run. **No fitting,
tuning, resampling for training, recalibration or threshold selection occurs here.**

The central comparison is eligible **Freddie 2018 Q1** versus actual **Fannie 2018 Q1 originations
in the available acquisition file**. Report Fannie primary and strict eligibility separately.
Score Freddie 2018–2023 half-year cohorts, with annual/quarterly context, using the same models.
Retain Notebook 16's development and 2017 selection results as clearly labelled context.
This is retrospective: complete 24-month development outcomes were not all available in January
2018, so these final models are not presented as deployable at that historical date.

The primary analysis uses 22 shared predictors. The 24-feature analysis uses the already constructed
minimum-disclosed Fannie score **as a sensitivity proxy**, not a certified identical Freddie score.
Different sample sizes are retained; Fannie is not reduced to a 50,000-loan sample.

Open a fresh kernel in the project `.venv`, then **Run All**. Keep the reviewed Notebook 16 output
folder intact. Generated individual predictions remain local; upload only the final aggregate ZIP.

## 1. Freeze the evaluation interpretation and verify the reviewed run

Use the existing first-payment months 0–23 composite. Freddie includes numeric serious delinquency,
RA and selected terminal codes; Fannie uses numeric serious delinquency and selected terminal codes.
Codes 03, 09 and 15 have agency-specific meanings. The comparison is of mapped composite outcomes,
not proof of identical labels or a pure agency effect. Do not evaluate a numeric-only label using
these composite-trained models.

Keep both Fannie eligibility policies already constructed in Notebook 15. Strict excludes required
reporting gaps, unknown active status and in-window repurchases. Freddie retains its original eligibility
and HARP population; Fannie primary dataset coverage excludes HARP. No new population matching,
weighting, outlier filters, backfilling or conflict exclusions are added.

The original Fannie download date/release is unretained. The reviewed input hashes and positional/
mapping audits identify the data used. Missing download history alone does not block evaluation.
Earlier notebooks/manifests remain historical records and are not rewritten to change their status.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, sys, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, xgboost, joblib
from sklearn.metrics import roc_curve, precision_recall_curve
from IPython.display import display

ROOT = (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()).resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.freddie_features import REQUIRED_SOURCE_COLUMNS
from src.freddie_companion import (
    TARGET, SPECIFICATIONS, require, build_companion_features, apply_contract, calibration_table,
)
from src.fannie_external_evaluation import (
    METRICS, MetricCache, cohort_metadata, input_coverage, summarize,
    bootstrap_group, intervals, paired_model_intervals, stable_seed,
)

DATA = ROOT / 'data/processed/Freddie_Mac'
FANNIE = ROOT / 'data/processed/Fannie_Mae'
REVIEWED_RUN_ID = '20261125T142258634663Z'
REVIEWED_MANIFEST_SHA256 = '0831ffa3f152a46c0d6745cec758fd3dc98bf6b416dc57b3d9ccb8461afbfac7'
COMPANION_RUN = DATA / 'companion_models_2015_2017' / REVIEWED_RUN_ID
FANNIE_RUN = None  # Optional exact Notebook 15 folder; normally discovered by its reviewed metadata hash.
N_BOOTSTRAP = 1000
SEED = 42
SOURCE_BASELINE_COMMIT = '9186d0f1a36a578221430c5a03e478ef477283c3'
EXPECTED_FREDDIE_OOT = 294070
EXPECTED_FREDDIE_Q1 = 11884
EXPECTED_FANNIE = dict(total=260395, primary=260244, strict=260043,
                       primary_events=1960, strict_events=1947, conflicts=56)
OUT = FANNIE / 'notebook_17_external_model_reliability' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')

def sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def notebook_source_sha(path):
    n = json.loads(Path(path).read_text(encoding='utf-8'))
    payload = [(c['cell_type'], ''.join(c['source'])) for c in n['cells']]
    return hashlib.sha256(json.dumps(payload, ensure_ascii=False).encode()).hexdigest()

def local_relative(name):
    relative = Path(str(name).replace(chr(92), '/'))
    require(not relative.is_absolute() and '..' not in relative.parts, 'Invalid stored relative path.')
    return ROOT / relative

manifest_path = COMPANION_RUN / 'model_manifest.json'
require(manifest_path.is_file(), f'Missing reviewed Notebook 16 folder: {COMPANION_RUN}')
require(sha(manifest_path) == REVIEWED_MANIFEST_SHA256, 'This is not the reviewed Notebook 16 manifest.')
m16 = json.loads(manifest_path.read_text(encoding='utf-8'))
require(m16['run_id'] == REVIEWED_RUN_ID and m16['stage'] == '16', 'Reviewed run identity differs.')
require(m16['status'] == 'development_complete_pending_review' and m16['original_artifacts_preserved'],
        'Notebook 16 run is incomplete or original artifacts changed.')
require(m16['target'] == TARGET and m16['horizon_months'] == 24, 'Companion target differs.')
require(m16['specifications'] == SPECIFICATIONS and m16['development_rows'] == 143598
        and m16['development_events'] == 963, 'Reviewed development specification differs.')
require(m16['development_years'] == [2015, 2016, 2017]
        and m16['internal_fit_years'] == [2015, 2016] and m16['internal_validation_year'] == 2017,
        'Development split differs.')
require(not m16['external_scored'] and not m16['fannie_outcomes_loaded'] and not m16['out_of_time_rows_loaded'],
        'Expected the reviewed development-only run.')
require(N_BOOTSTRAP == 1000, 'The prespecified production bootstrap uses 1,000 draws.')
versions = dict(python=platform.python_version(), pandas=pd.__version__, numpy=np.__version__,
                scikit_learn=sklearn.__version__, xgboost=xgboost.__version__, joblib=joblib.__version__)
for name in ['scikit_learn', 'xgboost', 'joblib', 'numpy', 'pandas']:
    require(versions[name] == m16[name], f'{name} differs from model-training environment; use the original .venv.')
protected = {str(manifest_path.relative_to(ROOT)): sha(manifest_path)}
for name, expected in m16['output_sha256'].items():
    relative = Path(name.replace(chr(92), '/'))
    require(not relative.is_absolute() and '..' not in relative.parts, 'Invalid Notebook 16 output path.')
    p = COMPANION_RUN / relative
    require(p.is_file() and sha(p) == expected, f'Notebook 16 artifact changed or missing: {relative}')
    protected[str(p.relative_to(ROOT))] = expected
for name, expected in m16['original_artifacts_sha256'].items():
    p = local_relative(name)
    require(p.is_file() and sha(p) == expected, f'Original artifact changed: {name}')
    protected[str(p.relative_to(ROOT))] = expected
for name, expected in m16['source_sha256'].items():
    require(sha(ROOT / name) == expected, f'Reviewed feature source changed: {name}')
    protected[name] = expected
for name in ['src/fannie_external_evaluation.py']:
    protected[name] = sha(ROOT / name)
notebook_path = ROOT / 'notebooks/17_fannie_external_model_reliability.ipynb'
notebook_source_before = notebook_source_sha(notebook_path)

possible = [Path(FANNIE_RUN) / 'metadata.json'] if FANNIE_RUN else sorted(
    (FANNIE / 'notebook_15_external_dataset_construction').glob('*/metadata.json'))
matches = [p for p in possible if p.is_file() and sha(p) == m16['metadata_15_sha256']]
require(bool(matches), 'Cannot find the reviewed Notebook 15 metadata. Set FANNIE_RUN to that folder.')
metadata15_path = matches[0]  # Identical copies with the reviewed hash are equivalent.
meta15 = json.loads(metadata15_path.read_text(encoding='utf-8'))
require(meta15['candidate_dataset_sha256'] == m16['fannie_candidate_sha256']
        and meta15['source_sha256'] == m16['fannie_source_sha256'], 'Fannie provenance differs.')
require(meta15['primary_features'] == SPECIFICATIONS['primary_22']
        and meta15['fico_sensitivity_features'] == SPECIFICATIONS['fico_sensitivity_24'], 'Fannie predictor order differs.')
require(meta15['models_scored'] is False and meta15['physical_rows'] == 23563399
        and meta15['unique_loans'] == 458830, 'Notebook 15 source identity differs.')
INPUT = DATA / 'freddie_modeling_dataset_2015_2023_24m.parquet'
CANDIDATE = metadata15_path.parent / 'fannie_2018q1_candidate_dataset.parquet'
require(sha(INPUT) == m16['input_file_sha256'], 'Freddie modeling input changed.')
require(sha(CANDIDATE) == m16['fannie_candidate_sha256'], 'Fannie candidate input changed.')
protected[str(INPUT.relative_to(ROOT))] = m16['input_file_sha256']
protected[str(CANDIDATE.relative_to(ROOT))] = m16['fannie_candidate_sha256']
protected[str(metadata15_path.relative_to(ROOT))] = m16['metadata_15_sha256']
model_keys = [f'{spec}__{family}' for spec in SPECIFICATIONS for family in ['logistic', 'xgboost']]
require(set(model_keys) == set(m16['models']), 'Four reviewed models are required.')
frozen_edges = json.loads((COMPANION_RUN / 'frozen_calibration_bins.json').read_text(encoding='utf-8'))
require(set(frozen_edges) == set(model_keys), 'Frozen calibration references differ.')
models, contracts = {}, {}
for key in model_keys:
    record = m16['models'][key]
    for name in [record['artifact'], record['feature_contract']]:
        require(Path(name).name == name, 'Invalid model/contract filename.')
    contract = json.loads((COMPANION_RUN / record['feature_contract']).read_text(encoding='utf-8'))
    require(contract['training_years'] == [2015, 2016, 2017]
            and contract['predictors'] == SPECIFICATIONS[record['specification']], 'Frozen model contract differs.')
    pipe = joblib.load(COMPANION_RUN / record['artifact'])  # Load only verified, researcher-created artifacts.
    require(list(pipe.feature_names_in_) == record['predictors'], 'Serialized model input order differs.')
    for name, value in record['selected_params'].items():
        require(pipe.named_steps['model'].get_params()[name] == value, 'Serialized model settings differ.')
    models[key], contracts[key] = pipe, contract
OUT.mkdir(parents=True, exist_ok=False)
(OUT / 'figures').mkdir()
print('Loaded exactly the four reviewed models; no fitting will occur.')
print('Output folder:', OUT)

## 2. Score Freddie 2018–2023 without changing preprocessing

Read only existing eligible modeling rows from 2018 onward. Rebuild the same companion features,
apply each final development contract, and call `predict_proba`. The 2018 Q1 comparison is a subset
of these predictions. The original 29-feature models are not rescored or modified.

In [ ]:
columns = list(dict.fromkeys([*REQUIRED_SOURCE_COLUMNS, 'HORIZON_MONTHS']))
freddie_raw = pd.read_parquet(INPUT, engine='pyarrow', columns=columns,
                              filters=[('SOURCE YEAR', 'in', list(range(2018, 2024)))])
require(set(freddie_raw['SOURCE YEAR'].unique()) == set(range(2018, 2024)), 'OOT row filter differs.')
require(len(freddie_raw) == EXPECTED_FREDDIE_OOT, 'OOT population count differs.')
require(freddie_raw.HORIZON_MONTHS.eq(24).all(), 'Freddie horizon differs.')
cohorts = cohort_metadata(freddie_raw)
require(set(cohorts.quarter) == {f'{y}Q{q}' for y in range(2018, 2024) for q in range(1, 5)},
        'Freddie quarterly coverage differs.')
require(int(cohorts.quarter.eq('2018Q1').sum()) == EXPECTED_FREDDIE_Q1, 'Matched Freddie Q1 count differs.')
freddie_x = build_companion_features(freddie_raw)
freddie = pd.DataFrame(dict(loan_id=freddie_raw['LOAN IDENTIFIER'].astype('string'),
                            y_true=freddie_x[TARGET].to_numpy(dtype=int),
                            year=cohorts.year, quarter=cohorts.quarter, half_year=cohorts.half_year))
coverage_rows = []
for key in model_keys:
    x = apply_contract(freddie_x, contracts[key])
    p = models[key].predict_proba(x)[:, 1].astype(float)
    require(np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all(), f'Invalid Freddie predictions: {key}')
    freddie[key] = p
    for period in sorted(freddie.half_year.unique()):
        mask = freddie.half_year.eq(period)
        coverage_rows.append(input_coverage(freddie_x.loc[mask], contracts[key], models[key]).assign(
            population=f'Freddie {period}', model=key))
    mask = freddie.quarter.eq('2018Q1')
    coverage_rows.append(input_coverage(freddie_x.loc[mask], contracts[key], models[key]).assign(
        population='Freddie 2018Q1', model=key))
require(not freddie[model_keys].isna().any().any(), 'Incomplete Freddie prediction coverage.')
freddie.to_parquet(OUT / 'freddie_2018_2023_predictions.parquet', index=False)
del freddie_x, freddie_raw
print('Freddie OOT loans scored:', len(freddie))

## 3. Score the reviewed Fannie population once

Use the earliest-snapshot features already saved by Notebook 15. Score primary eligible loans;
strict is a nested subset using identical probabilities, not a separate fitted or recalibrated model.
Retain the 56 flagged conflicting histories, as reviewed. Unknown or missing inputs follow the
stored Freddie rules. The raw Fannie CSV is not read or rescanned.

In [ ]:
fannie_raw = pd.read_parquet(CANDIDATE)
required = ['loan_id', 'orig', 'ORIGINATION_COHORT', 'primary', 'strict',
            'TARGET_PRIMARY_COMPOSITE', 'TARGET_STRICT_COMPOSITE',
            'origination_field_conflict_count', 'MI_MAPPING_INCONSISTENT'] + SPECIFICATIONS['fico_sensitivity_24']
require(set(required).issubset(fannie_raw.columns), 'Fannie candidate columns missing.')
require(len(fannie_raw) == EXPECTED_FANNIE['total'] and fannie_raw.loan_id.notna().all()
        and fannie_raw.loan_id.is_unique, 'Fannie population/key count differs.')
dates = pd.to_datetime(fannie_raw.orig.astype('string'), format='%m%Y', errors='raise')
require(dates.dt.year.eq(2018).all() and dates.dt.quarter.eq(1).all()
        and fannie_raw.ORIGINATION_COHORT.eq('2018Q1').all(), 'Fannie actual origination quarter differs.')
for flag in ['primary', 'strict']:
    require(pd.api.types.is_bool_dtype(fannie_raw[flag]) and fannie_raw[flag].notna().all(), f'Invalid {flag} flag.')
require((~fannie_raw.strict | fannie_raw.primary).all(), 'Strict is not a subset of primary.')
for policy in ['primary', 'strict']:
    mask = fannie_raw[policy]
    labels = fannie_raw[f'TARGET_{policy.upper()}_COMPOSITE']
    require(labels.loc[mask].notna().all() and labels.loc[mask].isin([0, 1]).all()
            and labels.loc[~mask].isna().all(), f'Fannie {policy} label/eligibility differs.')
    require(int(mask.sum()) == EXPECTED_FANNIE[policy]
            and int(labels.loc[mask].sum()) == EXPECTED_FANNIE[f'{policy}_events'], f'Fannie {policy} counts differ.')
require(not fannie_raw.MI_MAPPING_INCONSISTENT.any(), 'MI mapping conflicts require review.')
require(int(fannie_raw.origination_field_conflict_count.gt(0).sum()) == EXPECTED_FANNIE['conflicts'],
        'Reviewed conflict count differs.')
pd.testing.assert_series_equal(fannie_raw.loc[fannie_raw.strict, 'TARGET_PRIMARY_COMPOSITE'],
                               fannie_raw.loc[fannie_raw.strict, 'TARGET_STRICT_COMPOSITE'], check_names=False)
for name in ['ORIGINAL_DTI', 'ORIGINAL_LTV', 'MI_PERCENT', 'FICO_SCORE']:
    require(fannie_raw[name + '_MISSING'].eq(fannie_raw[name].isna().astype(int)).all(),
            f'Fannie missingness flag differs: {name}')
eligible = fannie_raw.loc[fannie_raw.primary].copy().reset_index(drop=True)
fannie = pd.DataFrame(dict(loan_id=eligible.loan_id.astype('string'),
                           y_true=eligible.TARGET_PRIMARY_COMPOSITE.to_numpy(dtype=int),
                           strict=eligible.strict.to_numpy(dtype=bool),
                           y_strict=eligible.TARGET_STRICT_COMPOSITE,
                           origination_field_conflict_count=eligible.origination_field_conflict_count))
for key in model_keys:
    x = apply_contract(eligible, contracts[key])
    p = models[key].predict_proba(x)[:, 1].astype(float)
    require(np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all(), f'Invalid Fannie predictions: {key}')
    fannie[key] = p
    for policy, mask in [('primary', np.ones(len(eligible), dtype=bool)), ('strict', eligible.strict.to_numpy(dtype=bool))]:
        coverage_rows.append(input_coverage(eligible.loc[mask], contracts[key], models[key]).assign(
            population=f'Fannie 2018Q1 {policy}', model=key))
fannie.to_parquet(OUT / 'fannie_2018q1_predictions.parquet', index=False)
pd.concat(coverage_rows, ignore_index=True).to_csv(OUT / 'input_coverage.csv', index=False)
del eligible, fannie_raw
print('Fannie primary scored:', len(fannie), '| strict subset:', int(fannie.strict.sum()))

## 4. Point estimates and frozen-bin calibration

Report ROC-AUC, **Average Precision** (not trapezoidal PR-AUC), Brier, prevalence, mean predicted
risk, signed/absolute risk gap and two explicitly named ECE summaries. **Risk gap is predicted minus
observed**: negative means underprediction. This follows Notebook 16; the original Notebook 11
used the opposite sign, so do not merge those columns without converting their definition.

The main calibration curves and `ece_development_bins` use each final model's frozen development
prediction bins. `ece_fixed_10` retains Notebook 16's ten equal-width probability ranges. Both
are descriptive bin-dependent summaries. A small ECE does not guarantee good calibration in every group.

In [ ]:
groups = {}
group_info = {}
for kind, column in [('half_year', 'half_year'), ('annual', 'year'), ('quarter', 'quarter')]:
    for value in sorted(freddie[column].unique()):
        name = f'Freddie {value}'
        frame = freddie.loc[freddie[column].eq(value)].reset_index(drop=True)
        groups[name] = frame
        group_info[name] = dict(agency='Freddie', period_kind=kind, period=str(value), eligibility='original')
groups['Fannie 2018Q1 primary'] = fannie
groups['Fannie 2018Q1 strict'] = fannie.loc[fannie.strict].reset_index(drop=True)
for policy in ['primary', 'strict']:
    group_info[f'Fannie 2018Q1 {policy}'] = dict(agency='Fannie', period_kind='quarter', period='2018Q1', eligibility=policy)
point_parts, calibration_parts, fixed_parts = [], [], []
point_vectors = {}
for name, frame in groups.items():
    y = frame.y_true.to_numpy(dtype=int)
    pred = {key: frame[key].to_numpy(dtype=float) for key in model_keys}
    summary = summarize(y, pred, frozen_edges)
    summary['population'] = name
    for c, value in group_info[name].items():
        summary[c] = value
    point_parts.append(summary)
    point_vectors[name] = summary.set_index('model').loc[model_keys, METRICS].to_numpy(dtype=float)
    for key in model_keys:
        calibration_parts.append(calibration_table(y, pred[key], frozen_edges[key]).assign(population=name, model=key))
        fixed_parts.append(calibration_table(y, pred[key]).assign(population=name, model=key))
performance = pd.concat(point_parts, ignore_index=True)
calibration = pd.concat(calibration_parts, ignore_index=True)
fixed_calibration = pd.concat(fixed_parts, ignore_index=True)
central_names = ['Freddie 2018Q1', 'Fannie 2018Q1 primary', 'Fannie 2018Q1 strict']
central = performance.loc[performance.population.isin(central_names)].copy()
performance.to_csv(OUT / 'performance_point_estimates.csv', index=False)
central.to_csv(OUT / 'matched_2018q1_performance.csv', index=False)
calibration.to_csv(OUT / 'calibration_development_bins.csv', index=False)
fixed_calibration.to_csv(OUT / 'calibration_fixed_probability_bins.csv', index=False)
populations = performance.drop_duplicates('population')[['population', 'agency', 'period_kind', 'period',
                                                        'eligibility', 'n', 'events', 'negatives', 'prevalence']]
populations.to_csv(OUT / 'population_summary.csv', index=False)
context_parts = []
for filename, label in [('development_reference_metrics.csv', 'development_in_sample'),
                         ('development_2017_selected_metrics.csv', '2017_internal_selection')]:
    context = pd.read_csv(COMPANION_RUN / filename, float_precision='round_trip')
    context['model'] = context.specification + '__' + context.family
    context = context.rename(columns={'risk_gap': 'risk_gap_pred_minus_observed'})
    context['context'] = label
    context_parts.append(context[['context', 'model', 'n', 'events', 'prevalence', 'mean_predicted',
                                  'roc_auc', 'average_precision', 'brier', 'risk_gap_pred_minus_observed', 'ece_fixed_10']])
pd.concat(context_parts, ignore_index=True).to_csv(OUT / 'development_context.csv', index=False)
display(central[['population', 'model', 'n', 'events', 'prevalence', 'roc_auc', 'average_precision',
                 'brier', 'mean_predicted', 'risk_gap_pred_minus_observed', 'ece_development_bins']].round(6))

## 5. Loan-bootstrap uncertainty and paired comparisons

Use **1,000 ordinary, unstratified loan resamples** per Freddie half-year, the matched Freddie Q1,
and the parent Fannie primary population. All four models share each population's resampled loans.
Derive Fannie strict from the same primary resample: strict draw size varies, preserving dependence.
Fannie-minus-Freddie Q1 comparisons combine independently sampled agency populations.

Freeze prediction ordering once, then evaluate exact resample multiplicities, including tied scores.
This gives the same metrics as expanding the resampled loan rows, with no repeated probability sorting.
Invalid ranking draws with a missing outcome class are counted explicitly. Intervals are 95% percentile
intervals, conditional on the frozen fitted models and observed datasets. They assume independent loans
within a population and do not account for common economic shocks, sampling design, label-definition
uncertainty or model-training uncertainty. The intervals are marginal, without multiplicity adjustment.

Prespecified contrasts: XGBoost minus Logistic Regression within each feature specification;
FICO sensitivity minus primary within each model family; Fannie minus matched Freddie;
and strict minus primary Fannie. Do not choose specifications, policies or recalibration from these outcomes.
This cell may take several minutes and prints progress every 100 draws.

In [ ]:
bootstrap = {}
bootstrap_sizes = {}
bootstrap_names = [f'Freddie {year}-H{half}' for year in range(2018, 2024) for half in [1, 2]] + ['Freddie 2018Q1']
for name in bootstrap_names + ['Fannie 2018Q1 primary']:
    frame = groups[name]
    pred = {key: frame[key].to_numpy(dtype=float) for key in model_keys}
    is_fannie = name == 'Fannie 2018Q1 primary'
    result = bootstrap_group(frame.y_true.to_numpy(dtype=int), pred, frozen_edges,
                             draws=N_BOOTSTRAP, seed=stable_seed(name, SEED),
                             subset=frame.strict.to_numpy(dtype=bool) if is_fannie else None,
                             progress=lambda done, total, label=name: print(f'{label}: {done}/{total}', flush=True))
    bootstrap[name] = dict(models=result['models'], draws=result['draws'])
    bootstrap_sizes[name] = np.full(N_BOOTSTRAP, len(frame), dtype=int)
    if is_fannie:
        name_strict = 'Fannie 2018Q1 strict'
        bootstrap[name_strict] = dict(models=result['models'], draws=result['subset_draws'])
        bootstrap_sizes[name_strict] = result['subset_sizes']
interval_parts, paired_parts, draw_parts = [], [], []
for name, result in bootstrap.items():
    point = point_vectors[name]
    for j, key in enumerate(model_keys):
        interval_parts.append(intervals(result['draws'][:, j], point[j], dict(population=name, model=key)))
        draws = pd.DataFrame(result['draws'][:, j], columns=METRICS)
        draws['draw_index'] = np.arange(N_BOOTSTRAP)
        draws['n_draw'] = bootstrap_sizes[name]
        draws['population'] = name
        draws['model'] = key
        draw_parts.append(draws)
    paired_parts.append(paired_model_intervals(result, point, name))
uncertainty = pd.concat(interval_parts, ignore_index=True)
paired = pd.concat(paired_parts, ignore_index=True)
cross_parts = []
for policy in ['primary', 'strict']:
    left, right = f'Fannie 2018Q1 {policy}', 'Freddie 2018Q1'
    for j, key in enumerate(model_keys):
        cross_parts.append(intervals(bootstrap[left]['draws'][:, j] - bootstrap[right]['draws'][:, j],
                                    point_vectors[left][j] - point_vectors[right][j],
                                    dict(model=key, comparison=f'{left}_minus_{right}', dependence='independent_agency_populations')))
for j, key in enumerate(model_keys):
    left, right = 'Fannie 2018Q1 strict', 'Fannie 2018Q1 primary'
    cross_parts.append(intervals(bootstrap[left]['draws'][:, j] - bootstrap[right]['draws'][:, j],
                                point_vectors[left][j] - point_vectors[right][j],
                                dict(model=key, comparison='Fannie_strict_minus_primary', dependence='nested_same_parent_resample')))
cross_population = pd.concat(cross_parts, ignore_index=True)
uncertainty.to_csv(OUT / 'bootstrap_metric_intervals.csv', index=False)
paired.to_csv(OUT / 'paired_model_difference_intervals.csv', index=False)
cross_population.to_csv(OUT / 'cross_population_difference_intervals.csv', index=False)
# These are aggregate metric draws; no sampled identifiers or individual predictions are exported.
pd.concat(draw_parts, ignore_index=True).to_csv(OUT / 'bootstrap_metric_draws.csv', index=False)
display(uncertainty.loc[uncertainty.population.isin(central_names)
                       & uncertainty.metric.isin(['roc_auc', 'average_precision', 'brier', 'ece_development_bins'])].round(6))

## 6. Plot the matched comparison and Freddie temporal context

Use frozen development bins for calibration. PR plots include each population's prevalence line;
AP and Brier depend on prevalence, so unequal values are not a pure measure of agency transfer.
The half-year plots show the companion models' retrospective OOT context. This single Fannie
acquisition-file subset cannot validate the full original drift, warning or retraining experiment.

In [ ]:
colors = {'Freddie 2018Q1': '#1f77b4', 'Fannie 2018Q1 primary': '#d95f02', 'Fannie 2018Q1 strict': '#2ca02c'}
for key in model_keys:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for name in central_names:
        frame = groups[name]
        y, p = frame.y_true.to_numpy(dtype=int), frame[key].to_numpy(dtype=float)
        line_style = '--' if name.endswith('strict') else '-'
        if len(np.unique(y)) == 2:
            fpr, tpr, _ = roc_curve(y, p)
            precision, recall, _ = precision_recall_curve(y, p)
            axes[0].plot(fpr, tpr, color=colors[name], linestyle=line_style, label=name)
            axes[1].plot(recall, precision, color=colors[name], linestyle=line_style, label=name)
        axes[1].axhline(float(y.mean()), color=colors[name], linestyle=':', alpha=.5)
        bins = calibration.loc[calibration.population.eq(name) & calibration.model.eq(key) & calibration.n.gt(0)]
        axes[2].plot(bins.mean_predicted, bins.observed_rate, marker='o', color=colors[name],
                     linestyle=line_style, label=name)
    axes[0].plot([0, 1], [0, 1], '--', color='gray')
    axes[0].set(title='ROC curve', xlabel='False positive rate', ylabel='True positive rate')
    axes[1].set(title='Precision–recall; dotted lines = prevalence', xlabel='Recall', ylabel='Precision', ylim=(0, 1.02))
    bounds = calibration.loc[calibration.population.isin(central_names) & calibration.model.eq(key),
                              ['mean_predicted', 'observed_rate']].max().max()
    upper = max(.01, float(bounds))
    axes[2].plot([0, upper], [0, upper], '--', color='gray')
    axes[2].set(title='Frozen development-bin calibration', xlabel='Mean predicted risk', ylabel='Observed event rate')
    axes[0].legend(fontsize=8)
    fig.suptitle(key + ' — matched 2018 Q1 comparison')
    fig.tight_layout()
    fig.savefig(OUT / 'figures' / f'matched_2018q1_{key}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
periods = [f'{year}-H{half}' for year in range(2018, 2024) for half in [1, 2]]
plot_metrics = [('roc_auc', 'ROC-AUC'), ('average_precision', 'Average Precision'),
                ('brier', 'Brier score'), ('absolute_risk_gap', 'Absolute risk gap')]
for spec in SPECIFICATIONS:
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for axis, (metric, title) in zip(axes.flat, plot_metrics):
        for family, color in [('logistic', '#1f77b4'), ('xgboost', '#d95f02')]:
            key = f'{spec}__{family}'
            data = performance.loc[performance.model.eq(key) & performance.period_kind.eq('half_year')]
            data = data.set_index('period').loc[periods]
            ci = uncertainty.loc[uncertainty.model.eq(key) & uncertainty.metric.eq(metric)]
            ci = ci.set_index('population').loc[[f'Freddie {p}' for p in periods]]
            axis.plot(np.arange(12), data[metric], marker='o', label=family, color=color)
            axis.fill_between(np.arange(12), ci.ci_low, ci.ci_high, color=color, alpha=.15)
        axis.set(title=title, xticks=np.arange(12), xticklabels=periods)
        axis.tick_params(axis='x', rotation=60)
        axis.legend()
    fig.suptitle(spec + ' — Freddie OOT context; marginal 95% loan-bootstrap intervals')
    fig.tight_layout()
    fig.savefig(OUT / 'figures' / f'freddie_half_year_{spec}.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

## 7. Reconcile exports and save the aggregate review ZIP

Local prediction Parquets support later audits but are excluded from the ZIP. Aggregate bootstrap
draws are included so interval calculations and paired differences can be independently checked.
The final manifest records exact input/artifact hashes and confirms that the original files and
reviewed companion artifacts remain unchanged. A folder without this final manifest is incomplete.

In [ ]:
require(len(central) == 12 and len(performance) == (12 + 6 + 24 + 2) * 4, 'Incomplete performance coverage.')
for name, frame in groups.items():
    part = performance.loc[performance.population.eq(name)]
    require(len(part) == 4 and part.n.eq(len(frame)).all() and part.events.eq(frame.y_true.sum()).all(),
            f'Performance population mismatch: {name}')
    for key in model_keys:
        part = calibration.loc[calibration.population.eq(name) & calibration.model.eq(key)]
        require(int(part.n.sum()) == len(frame) and int(part.events.sum()) == int(frame.y_true.sum()),
                f'Calibration totals differ: {name}/{key}')
require({name: sha(ROOT / name) for name in protected} == protected, 'An input or protected artifact changed during evaluation.')
require(notebook_source_sha(notebook_path) == notebook_source_before, 'Notebook source changed during the run.')
protocol = dict(
    target=TARGET, horizon_months=24,
    outcome_interpretation='Mapped agency-specific composite; Freddie RA differs from Fannie unknown XX.',
    central_populations=central_names, primary_specification='primary_22',
    sensitivity_specification='fico_sensitivity_24; minimum-disclosed Fannie score proxy',
    fannie_eligibility=['primary', 'strict_nested_subset'],
    freddie_eligibility='Original modeling eligibility; original HARP population retained.',
    no_new_population_filters=True, no_reweighting=True, no_recalibration=True, no_threshold_selection=True,
    metric_definitions=dict(average_precision='Non-interpolated sklearn-compatible AP, not trapezoidal PR-AUC.',
                            risk_gap_pred_minus_observed='Mean predicted probability minus observed event rate.',
                            ece_development_bins='Weighted absolute bin gaps using final-model development prediction bins.',
                            ece_fixed_10='Ten fixed equal-width probability ranges, consistent with Notebook 16.'),
    bootstrap=dict(draws=N_BOOTSTRAP, seed=SEED, method='Ordinary iid loan bootstrap via exact integer multiplicities.',
                   intervals='Marginal 95% percentile; no multiplicity adjustment.',
                   paired_models=True, fannie_strict='Derived from same primary draw; subset draw size varies.',
                   independent_agencies=True, fit_uncertainty_included=False,
                   common_shocks_and_sampling_design_uncertainty_included=False),
    no_numeric_only_outcome_evaluation=True, fannie_download_date='not_retained',
    fannie_publication_release='unverified_provenance_note',
)
(OUT / 'evaluation_protocol.json').write_text(json.dumps(protocol, indent=2, allow_nan=False), encoding='utf-8')
notes = """# Notebook 17 review

Four frozen companion models were evaluated without model updates. The main comparison is
Freddie versus Fannie actual 2018 Q1 originations; primary and nested strict Fannie eligibility
are separate. FICO remains a proxy sensitivity, not an established identical agency score.

AP and Brier depend on prevalence. Risk gap here is predicted minus observed; this is the
opposite sign of the original Notebook 11 risk-gap column. Development references are in-sample,
and 2017 results were used for model selection. Neither is an independent external test result.

Confidence intervals condition on fitted models and assume independent loans. Primary/strict
Fannie intervals and differences preserve nesting. They do not resolve population/product coverage,
agency outcome semantics, common economic shocks, sampling design or feature publication timing.

Do not select new features, eligibility, thresholds or recalibration using these outcomes.
One acquisition-file subset cannot validate the complete original temporal/drift/retraining study.
The original download date is unretained; source-file hashes and validation evidence are recorded.
No original artifact or historical execution manifest was changed. No synthesis conclusion is
automatically approved by executing this notebook. Review this bundle before Notebook 18.
"""
(OUT / 'review_notes.md').write_text(notes, encoding='utf-8')
export_files = sorted(p for p in OUT.rglob('*') if p.is_file())
output_hashes = {str(p.relative_to(OUT)).replace(chr(92), '/'): sha(p) for p in export_files}
manifest = dict(stage='17', status='evaluation_complete_pending_interpretation',
                created_utc=datetime.now(timezone.utc).isoformat(),
                source_baseline_commit=SOURCE_BASELINE_COMMIT, notebook_source_sha256=notebook_source_before,
                reviewed_companion_run=REVIEWED_RUN_ID, reviewed_companion_manifest_sha256=REVIEWED_MANIFEST_SHA256,
                input_and_protected_artifacts_sha256={name.replace(chr(92), '/'): value for name, value in protected.items()},
                fannie_source_sha256=m16['fannie_source_sha256'], candidate_sha256=m16['fannie_candidate_sha256'],
                model_artifact_sha256={key: item['sha256'] for key, item in m16['models'].items()},
                original_artifacts_preserved=True, companion_artifacts_preserved=True,
                models_fitted=False, models_updated=False, external_scored=True,
                fannie_primary_loans=len(fannie), fannie_strict_loans=int(fannie.strict.sum()),
                freddie_oot_loans=len(freddie), bootstrap_draws=N_BOOTSTRAP,
                strict_bootstrap_n_min=int(bootstrap_sizes['Fannie 2018Q1 strict'].min()),
                strict_bootstrap_n_max=int(bootstrap_sizes['Fannie 2018Q1 strict'].max()),
                versions=versions, output_sha256=output_hashes)
(OUT / 'model_evaluation_manifest.json').write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding='utf-8')
review_zip = OUT / 'notebook_17_external_reliability_review.zip'
with zipfile.ZipFile(review_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.suffix in {'.csv', '.json', '.png', '.md'}:
            archive.write(p, str(p.relative_to(OUT)).replace(chr(92), '/'))
with zipfile.ZipFile(review_zip) as archive:
    require(archive.testzip() is None, 'Review ZIP failed integrity checks.')
    require(not any(name.endswith(('.parquet', '.joblib', '.sqlite')) for name in archive.namelist()),
            'Local-only artifact entered the review ZIP.')
print('Evaluation complete. Original and companion artifacts preserved.')
print('Keep individual predictions local:', OUT)
print('UPLOAD THIS FILE:', review_zip)

## Next review

Save this executed notebook and upload **`notebook_17_external_reliability_review.zip`** from the
printed folder. No earlier notebook needs to be rerun. Keep prediction Parquets and the exact
Notebook 16 artifacts local for audit. Review the matched comparison, uncertainty, calibration,
input coverage and sensitivities before writing the final cross-agency interpretation in Notebook 18.